In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from lightgbm import LGBMRegressor


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
Q1_path = os.path.join(path, 'Q1_data.csv')
Q1_data = pd.read_csv(Q1_path)


In [ ]:
# Task 2: Write your code here:
Q1_data.head()

In [ ]:
# Task 3: Write your code here:
Q1_data.info()
# there are some nulls in the data as you can see

In [ ]:
# Task 4: Write your code here:
Q1_data.describe()

In [ ]:
# Task 5: Write your code here:
Q1_data["Delivery_Time"].hist()

In [ ]:
# Task 1: Write your code here:
clean_data = Q1_data.drop("Order_ID", axis=1)
clean_data

In [ ]:
# Task 2: Write your code here:
# we split the numeric data then the categorical data then fill with mean and mode respectivly
categorical_cols = clean_data.select_dtypes(include=["object"]).columns
num_cols = clean_data.select_dtypes(include=["int64", "float64"]).columns

for col in categorical_cols:
    clean_data[col] = clean_data[col].fillna(clean_data[col].mode()[0])
for col in num_cols:
    clean_data[col] = clean_data[col].fillna(clean_data[col].mean())

clean_data.info()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(clean_data)
clean_data.info()
# there was dublicate but i ran the code twice by mistake because i wanted to see the number of sampels

In [ ]:
# Task 4: Write your code here:


for col in categorical_cols:

  le = LabelEncoder()
  clean_data[col] = le.fit_transform(clean_data[col].astype(str))

clean_data.head()

In [ ]:
features_cols = clean_data.select_dtypes(include=["int64", "float64", "object"]).columns.drop("Delivery_Time")
print(features_cols)

In [ ]:
# Task 5: Write your code here:
features_cols = clean_data.select_dtypes(include=["int64", "float64", "object"]).columns.drop("Delivery_Time")
scaler = StandardScaler()
clean_data[features_cols] = scaler.fit_transform(clean_data[features_cols])

clean_data.head()
# as you can see i scaled everything exept the target

In [ ]:
# Task 6: Write your code here:
# the target is continus so there is no such thing as imbalnce

In [ ]:
# Task 1: Write your code here:
X = clean_data[features_cols]
y = clean_data["Delivery_Time"]

X.head()


In [ ]:
# Task 2,3,4,5: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae_scores.append(mean_absolute_error(y_test, y_pred))


mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': features_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
y_pred_frame = pd.DataFrame({'Y_pred': y_pred})
y_pred_frame.hist()

In [ ]:
# Task Bonus: Write your code here:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores = []

model1 = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model2 = LGBMRegressor(verbose=-1)
# Iterate through folds
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]


    model1.fit(X_train, y_train)
    model2.fit(X_train, y_train)
    y_pred1 = model.predict(X_test)
    y_pred2 = model.predict(X_test)
    A_pred = (y_pred1 + y_pred2)/2
    mae_scores.append(mean_absolute_error(y_test, A_pred))


mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results for 2 models:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# the MAE is signefectly better since we are avraging the predection of 2 models (1-RandomForestRegressor   2- LGBMRegressor )